# Capítulo 4: Análisis Exploratorio de Datos y Modelo Base Predictivo

Esta libreta ejecuta el **Análisis Exploratorio de Datos (Subsección 4.1)** y el entrenamiento del **Modelo Base Predictivo (Subsección 4.2)** sobre los datos limpios y consolidados de la **Zona Gold** (`s3://american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee/metricas_distritales_trimestrales/`).

### Objetivos cubiertos según la guía académica UPAO:
1. **4.1 Análisis Exploratorio de Datos:** Evaluación sobre datos limpios respondiendo a tres preguntas estratégicas del negocio con tres visualizaciones cuantitativas:
   - Asimetría territorial en la velocidad de descarga móvil por departamento.
   - Relación demográfica poblacional frente a la dotación de estaciones base 4G instaladas.
   - Evolución temporal trimestral (2019-2024) contrastando distritos con alta infraestructura frente a distritos rezagados.
2. **4.2 Modelo Base:** Definición de la variable objetivo supervisada a un trimestre de anticipación ($t+1$), partición temporal cronológica estricta sin fuga de información, evaluación de la línea base ingenua (persistencia) y entrenamiento de un regresor de bosque aleatorio distribuido (`RandomForestRegressor` en Spark MLlib) para declarar formalmente la **Línea Base Técnica** de la Unidad 1.

### 1. Configuración del Entorno y Carga de Rutas del Data Lake

In [1]:
import os
import sys
import json
import time
from pathlib import Path

# Enlazar PySpark y dependencias de EMR
SPARK_HOME = Path("/usr/lib/spark")
if SPARK_HOME.is_dir():
    os.environ["SPARK_HOME"] = str(SPARK_HOME)
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    py4j_zips = sorted((SPARK_HOME / "python" / "lib").glob("py4j-*.zip"))
    if py4j_zips:
        sys.path.insert(0, str(py4j_zips[0]))
    sys.path.insert(0, str(SPARK_HOME / "python"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

BUCKET_GOLD = "american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee"
GOLD_URI = f"s3://{BUCKET_GOLD}/metricas_distritales_trimestrales/"
IMG_DIR = Path("/home/hadoop/dev/images")
IMG_DIR.mkdir(parents=True, exist_ok=True)

print("Configuración cargada exitosamente:")
print(f"  Gold S3: {GOLD_URI}")
print(f"  Directorio de imágenes: {IMG_DIR}")

Configuración cargada exitosamente:
  Gold S3: s3://american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee/metricas_distritales_trimestrales/
  Directorio de imágenes: /home/hadoop/dev/images


### 2. Inicialización de la Sesión Spark Distribuida en YARN

In [2]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, avg, count, sum as spark_sum, lead, round as spark_round,
    lit, when, desc, asc
)
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .master("yarn") \
    .appName("AmericanTower-EDA-y-ModeloBase") \
    .config("spark.submit.deployMode", "client") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

sc = spark.sparkContext
print(f"Sesión Spark activa: {spark.version} en modo YARN")
print(f"Paralelismo base:   {sc.defaultParallelism} particiones distribuidas")

Sesión Spark activa: 3.5.0-amzn-1 en modo YARN
Paralelismo base:   2 particiones distribuidas


### 3. Carga del Tablón Consolidado en S3 Gold (Datos Limpios)

In [3]:
print(f"Cargando tablón analítico Gold desde: {GOLD_URI}")
df_gold = spark.read.parquet(GOLD_URI).cache()
total_gold = df_gold.count()
print(f"Total registros limpios en Gold: {total_gold:,}")
df_gold.select("ubigeo", "departamento", "distrito", "year", "quarter", "total_eb_4g", "vel_descarga_mbps", "latencia_ms").show(5)

Cargando tablón analítico Gold desde: s3://american-tower-dev-gold-00f4f2b4cf37bac32a9ec520ee/metricas_distritales_trimestrales/
Total registros limpios en Gold: 45,360
+------+------------+--------+----+-------+-----------+-----------------+-----------+
|ubigeo|departamento|distrito|year|quarter|total_eb_4g|vel_descarga_mbps|latencia_ms|
+------+------------+--------+----+-------+-----------+-----------------+-----------+
|090203|HUANCAVELICA|    ANTA|2022|      4|          0|             13.9|       65.0|
|090203|HUANCAVELICA|    ANTA|2020|      3|          0|             10.3|       65.0|
|090203|HUANCAVELICA|    ANTA|2022|      3|          0|             13.9|       65.0|
|090203|HUANCAVELICA|    ANTA|2024|      1|          0|             17.5|       65.0|
|090203|HUANCAVELICA|    ANTA|2023|      1|          0|             15.7|       65.0|
+------+------------+--------+----+-------+-----------+-----------------+-----------+
only showing top 5 rows



### 4. Subsección 4.1: Análisis Exploratorio de Datos (Visualizaciones Obligatorias)

Se responden formalmente las tres preguntas analíticas de negocio formuladas en la guía docente:

In [4]:
print(">>> [Visualización 1/3] Calculando métricas departamentales...")
df_depto = df_gold.groupBy("departamento").agg(
    spark_round(avg("vel_descarga_mbps"), 2).alias("avg_descarga"),
    spark_round(avg("total_eb_4g"), 1).alias("avg_eb_4g"),
    spark_round(avg("latencia_ms"), 1).alias("avg_latencia")
).orderBy(desc("avg_descarga")).toPandas()

print("Top 5 Departamentos con mayor velocidad:")
print(df_depto.head(5).to_string(index=False))
print("\nDepartamentos con mayor rezago:")
print(df_depto.tail(5).to_string(index=False))

plt.figure(figsize=(10, 8))
colores = ["#003366" if v >= 30 else "#2b7bba" if v >= 20 else "#e05a47" for v in df_depto["avg_descarga"]]
plt.barh(df_depto["departamento"][::-1], df_depto["avg_descarga"][::-1], color=colores[::-1], edgecolor="black", alpha=0.85)
plt.axvline(x=df_depto["avg_descarga"].mean(), color="red", linestyle="--", linewidth=1.5, label=f"Media Nacional ({df_depto['avg_descarga'].mean():.1f} Mbps)")
plt.title("Velocidad Media de Descarga Móvil por Departamento (2019-2024)", fontsize=13, fontweight="bold", pad=15)
plt.xlabel("Velocidad Media de Descarga (Mbps)", fontsize=11)
plt.ylabel("Departamento", fontsize=11)
plt.grid(axis="x", linestyle=":", alpha=0.6)
plt.legend(loc="lower right")
plt.tight_layout()
img1_path = IMG_DIR / "eda_1_velocidad_departamentos.png"
plt.savefig(img1_path, dpi=300)
plt.close()
print(f"Gráfico 1 generado exitosamente en: {img1_path}")

>>> [Visualización 1/3] Calculando métricas departamentales...
Top 5 Departamentos con mayor velocidad:
departamento  avg_descarga  avg_eb_4g  avg_latencia
      CALLAO         72.71      127.9          26.6
        LIMA         49.07       57.1          47.9
  LAMBAYEQUE         32.96       39.5          53.1
       PIURA         24.99       21.8          58.5
     UCAYALI         24.89       21.6          58.5

Departamentos con mayor rezago:
departamento  avg_descarga  avg_eb_4g  avg_latencia
        PUNO         18.44        7.3          62.8
       PASCO         18.12        6.6          63.0
     HUANUCO         18.11        6.5          63.0
      LORETO         17.56        5.3         108.4
    AMAZONAS         17.43        5.0          63.5
Gráfico 1 generado exitosamente en: /home/hadoop/dev/images/eda_1_velocidad_departamentos.png


In [5]:
print(">>> [Visualización 2/3] Analizando relación Demográfica vs Infraestructura 4G...")
df_distrito = df_gold.groupBy("ubigeo", "departamento", "distrito").agg(
    avg("poblacion_estimada").alias("poblacion"),
    avg("total_eb_4g").alias("eb_4g")
).filter(col("poblacion") > 100).toPandas()

plt.figure(figsize=(10, 6))
plt.scatter(
    df_distrito["poblacion"], df_distrito["eb_4g"],
    alpha=0.45, color="#00558f", edgecolors="none", s=25
)
plt.xscale("log")
plt.title("Relación entre Población Distrital y Estaciones Base 4G Instaladas", fontsize=13, fontweight="bold", pad=15)
plt.xlabel("Población Distrital Estimada (Escala Logarítmica)", fontsize=11)
plt.ylabel("Cantidad de Estaciones Base 4G", fontsize=11)
plt.axhline(y=1, color="red", linestyle="--", alpha=0.7, label="Umbral crítico de déficit (≤ 1 Estación 4G)")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left")
plt.tight_layout()
img2_path = IMG_DIR / "eda_2_poblacion_vs_estaciones.png"
plt.savefig(img2_path, dpi=300)
plt.close()
print(f"Gráfico 2 generado exitosamente en: {img2_path}")

>>> [Visualización 2/3] Analizando relación Demográfica vs Infraestructura 4G...
Gráfico 2 generado exitosamente en: /home/hadoop/dev/images/eda_2_poblacion_vs_estaciones.png


In [6]:
print(">>> [Visualización 3/3] Calculando serie temporal de velocidad (2019-2024)...")
df_tiempo = df_gold.groupBy("year", "quarter").agg(
    spark_round(avg("vel_descarga_mbps"), 2).alias("promedio_nacional"),
    spark_round(avg(when(col("total_eb_4g") >= 5, col("vel_descarga_mbps"))), 2).alias("alta_infraestructura"),
    spark_round(avg(when(col("total_eb_4g") == 0, col("vel_descarga_mbps"))), 2).alias("rezagados_sin_4g")
).orderBy("year", "quarter").toPandas()

df_tiempo["periodo"] = df_tiempo["year"].astype(str) + "-Q" + df_tiempo["quarter"].astype(str)

plt.figure(figsize=(11, 6))
plt.plot(df_tiempo["periodo"], df_tiempo["alta_infraestructura"], marker="o", color="#008080", linewidth=2.2, label="Distritos con Alta Infraestructura (≥5 EB)")
plt.plot(df_tiempo["periodo"], df_tiempo["promedio_nacional"], marker="s", color="#003366", linewidth=2.2, linestyle="--", label="Promedio Nacional")
plt.plot(df_tiempo["periodo"], df_tiempo["rezagados_sin_4g"], marker="^", color="#d9534f", linewidth=2.2, label="Distritos Rezagados (0 EB 4G)")
plt.title("Evolución Trimestral de la Velocidad Media de Descarga Móvil (2019-2024)", fontsize=13, fontweight="bold", pad=15)
plt.xlabel("Período Trimestral", fontsize=11)
plt.ylabel("Velocidad Media de Descarga (Mbps)", fontsize=11)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="upper left")
plt.tight_layout()
img3_path = IMG_DIR / "eda_3_evolucion_temporal.png"
plt.savefig(img3_path, dpi=300)
plt.close()
print(f"Gráfico 3 generado exitosamente en: {img3_path}")

>>> [Visualización 3/3] Calculando serie temporal de velocidad (2019-2024)...
Gráfico 3 generado exitosamente en: /home/hadoop/dev/images/eda_3_evolucion_temporal.png


### 5. Subsección 4.2: Construcción y Evaluación del Modelo Base Predictivo en Spark MLlib

In [7]:
print(">>> Creando variable objetivo (Lead t+1)...")
window_spec = Window.partitionBy("ubigeo").orderBy("year", "quarter")
df_modelo = df_gold.withColumn("vel_descarga_t1", lead("vel_descarga_mbps", 1).over(window_spec))

df_supervisado = df_modelo.filter(col("vel_descarga_t1").isNotNull())
total_sup = df_supervisado.count()
print(f"  Total observaciones supervisadas con objetivo t+1: {total_sup:,}")

# Partición cronológica estricta para evitar data leakage
df_train = df_supervisado.filter(col("year") <= 2023).cache()
df_test = df_supervisado.filter(col("year") == 2024).cache()
train_count = df_train.count()
test_count = df_test.count()

print(f"  Partición temporal cronológica:")
print(f"    Conjunto de Entrenamiento (2019-2023): {train_count:,} filas ({(train_count/total_sup)*100:.1f} %)")
print(f"    Conjunto de Evaluación (2024):         {test_count:,} filas ({(test_count/total_sup)*100:.1f} %)")

>>> Creando variable objetivo (Lead t+1)...
  Total observaciones supervisadas con objetivo t+1: 43,470
  Partición temporal cronológica:
    Conjunto de Entrenamiento (2019-2023): 28,350 filas (65.2 %)
    Conjunto de Evaluación (2024):         7,560 filas (17.4 %)


In [8]:
print(">>> Evaluando Modelo Ingenuo (Persistencia: pred = vel_descarga_t)...")
eval_mae = RegressionEvaluator(labelCol="vel_descarga_t1", predictionCol="vel_descarga_mbps", metricName="mae")
eval_rmse = RegressionEvaluator(labelCol="vel_descarga_t1", predictionCol="vel_descarga_mbps", metricName="rmse")
eval_r2 = RegressionEvaluator(labelCol="vel_descarga_t1", predictionCol="vel_descarga_mbps", metricName="r2")

naive_mae = eval_mae.evaluate(df_test)
naive_rmse = eval_rmse.evaluate(df_test)
naive_r2 = eval_r2.evaluate(df_test)

print(f"  Línea Base Ingenua:")
print(f"    MAE:  {naive_mae:.2f} Mbps")
print(f"    RMSE: {naive_rmse:.2f} Mbps")
print(f"    R²:   {naive_r2:.4f}")

>>> Evaluando Modelo Ingenuo (Persistencia: pred = vel_descarga_t)...
  Línea Base Ingenua:
    MAE:  0.45 Mbps
    RMSE: 0.90 Mbps
    R²:   0.9989


In [9]:
print(">>> Entrenando RandomForestRegressor en Spark MLlib...")
feature_cols = [
    "vel_descarga_mbps", "vel_subida_mbps", "latencia_ms",
    "total_eb_4g", "total_eb_instaladas",
    "poblacion_estimada", "densidad_hab_km2",
    "tests", "devices", "quarter"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="vel_descarga_t1",
    numTrees=50,
    maxDepth=8,
    seed=42
)

pipeline = Pipeline(stages=[assembler, rf])
modelo_rf = pipeline.fit(df_train)

predicciones = modelo_rf.transform(df_test)

eval_mae_rf = RegressionEvaluator(labelCol="vel_descarga_t1", predictionCol="prediction", metricName="mae")
eval_rmse_rf = RegressionEvaluator(labelCol="vel_descarga_t1", predictionCol="prediction", metricName="rmse")
eval_r2_rf = RegressionEvaluator(labelCol="vel_descarga_t1", predictionCol="prediction", metricName="r2")

rf_mae = eval_mae_rf.evaluate(predicciones)
rf_rmse = eval_rmse_rf.evaluate(predicciones)
rf_r2 = eval_r2_rf.evaluate(predicciones)

print(f"  RandomForestRegressor (Spark MLlib):")
print(f"    MAE:  {rf_mae:.2f} Mbps")
print(f"    RMSE: {rf_rmse:.2f} Mbps")
print(f"    R²:   {rf_r2:.4f}")

rf_model_stage = modelo_rf.stages[-1]
importancias = rf_model_stage.featureImportances.toArray()
df_importancia = pd.DataFrame({
    "Caracteristica": feature_cols,
    "Importancia": importancias
}).sort_values("Importancia", ascending=False)

print("\nRanking de Importancia de Características:")
for idx, row in df_importancia.iterrows():
    print(f"  {row['Caracteristica']:<22}: {row['Importancia']*100:.2f} %")

>>> Entrenando RandomForestRegressor en Spark MLlib...
  RandomForestRegressor (Spark MLlib):
    MAE:  1.35 Mbps
    RMSE: 5.46 Mbps
    R²:   0.9603

Ranking de Importancia de Características:
  latencia_ms           : 28.60 %
  total_eb_4g           : 27.56 %
  vel_descarga_mbps     : 14.84 %
  vel_subida_mbps       : 11.07 %
  total_eb_instaladas   : 10.65 %
  devices               : 2.04 %
  tests                 : 1.80 %
  poblacion_estimada    : 1.78 %
  densidad_hab_km2      : 1.52 %
  quarter               : 0.14 %


### 6. Declaración Formal de la Línea Base Técnica para la Unidad 1

In [10]:
print("=" * 60)
print("SUBSECCIÓN 4.2: TABLA RESUMEN DE MODELOS Y LÍNEA BASE")
print("=" * 60)
print(f"| Modelo Evaluado                      | MAE (Mbps) | RMSE (Mbps) | R²     | Desempeño  |")
print(f"|--------------------------------------|------------|-------------|--------|------------|")
print(f"| Persistencia (Línea Base Ingenua)    | {naive_mae:10.2f} | {naive_rmse:11.2f} | {naive_r2:6.4f} | Referencia |")
print(f"| RandomForestRegressor (Spark MLlib)  | {rf_mae:10.2f} | {rf_rmse:11.2f} | {rf_r2:6.4f} | Explicativo|")
print("=" * 60)
print(f"LÍNEA BASE TÉCNICA AUDITADA: MAE = {rf_mae:.2f} Mbps | R² = {rf_r2:.4f}")
print("=" * 60)

SUBSECCIÓN 4.2: TABLA RESUMEN DE MODELOS Y LÍNEA BASE
| Modelo Evaluado                      | MAE (Mbps) | RMSE (Mbps) | R²     | Desempeño  |
|--------------------------------------|------------|-------------|--------|------------|
| Persistencia (Línea Base Ingenua)    |       0.45 |        0.90 | 0.9989 | Referencia |
| RandomForestRegressor (Spark MLlib)  |       1.35 |        5.46 | 0.9603 | Explicativo|
LÍNEA BASE TÉCNICA AUDITADA: MAE = 1.35 Mbps | R² = 0.9603
